In [2]:
import pandas as pd

На данном этапе мы хотим избавиться от дубликатов, сохранив только уникальные популярные книги из датасета с NYT. Они повторяются несколько раз, так как популярные книги входят в подборки популярных книг много раз. Оставим только те строки, где дата - максимальная, чтобы получить самые свежие по популярности уникальные книги

In [13]:
df = pd.read_excel('nyt_books.xlsx', index_col=0)

df.columns

Index(['age_group', 'amazon_product_url', 'article_chapter_link', 'asterisk',
       'author', 'book_image', 'book_image_height', 'book_image_width',
       'book_review_link', 'book_uri', 'contributor', 'contributor_note',
       'created_date', 'dagger', 'description', 'first_chapter_link', 'price',
       'primary_isbn10', 'primary_isbn13', 'publisher', 'rank',
       'rank_last_week', 'sunday_review_link', 'title', 'updated_date',
       'weeks_on_list', 'isbns', 'buy_links'],
      dtype='object')

In [78]:
df_prepared = df.sort_values('updated_date', ascending=False).drop_duplicates(subset=['title', 'author']).reset_index(drop=True)

In [79]:
df_prepared.head()

,age_group,amazon_product_url,article_chapter_link,asterisk,author,book_image,book_image_height,book_image_width,book_review_link,book_uri,...,primary_isbn13,publisher,rank,rank_last_week,sunday_review_link,title,updated_date,weeks_on_list,isbns,buy_links
0,NaN,https://www.amazon.com/dp/1638932239?tag=thene...,NaN,0,Navessa Allen,https://static01.nyt.com/bestsellers/images/97...,400,312,NaN,nyt://book/12c966bc-56fd-582c-b0cb-781e86d7079b,...,9781638932239,Slowburn,12,0,NaN,LIGHTS OUT,2026-04-03T16:35:10.385Z,8,"[{'isbn10': '', 'isbn13': '9781638932239'}]","[{'name': 'Amazon', 'url': 'https://www.amazon..."
1,NaN,https://www.amazon.com/dp/0804190119?tag=thene...,NaN,0,Timothy Snyder,NaN,0,0,NaN,nyt://book/eeb07cb6-f672-5b8d-b85d-6ce6f62b1856,...,9780804190114,Crown,3,2,NaN,ON TYRANNY,2026-04-02T01:29:16.596Z,39,"[{'isbn10': '', 'isbn13': '9780804190114'}]","[{'name': 'Amazon', 'url': 'https://www.amazon..."
2,NaN,https://www.amazon.com/dp/0593820282?tag=thene...,NaN,0,Matt Dinniman,https://static01.nyt.com/bestsellers/images/97...,500,331,NaN,nyt://book/b88e4106-56f5-5b57-8a3c-02a1afc2d346,...,9783748087342,Audible Studios,15,0,NaN,THE DUNGEON ANARCHIST'S COOKBOOK,2026-04-02T00:03:06.944Z,3,"[{'isbn10': '', 'isbn13': '9783748087342'}]","[{'name': 'Amazon', 'url': 'https://www.amazon..."
3,NaN,https://www.amazon.com/dp/0593820266?tag=thene...,NaN,0,Matt Dinniman,https://static01.nyt.com/bestsellers/images/97...,500,331,NaN,nyt://book/a757fdae-6248-5cd7-9f7e-a442615cd2e3,...,9783748087335,Audible Studios,13,0,NaN,CARL'S DOOMSDAY SCENARIO,2026-04-02T00:02:44.208Z,4,"[{'isbn10': '', 'isbn13': '9783748087335'}]","[{'name': 'Amazon', 'url': 'https://www.amazon..."
4,NaN,https://www.amazon.com/dp/B0F74CH248?tag=thene...,NaN,0,Matt Dinniman,https://static01.nyt.com/bestsellers/images/97...,500,331,NaN,nyt://book/9364bd7b-0ece-5625-87ae-c57d9f3446fa,...,9798350430479,Audible Studios,15,0,NaN,THIS INEVITABLE RUIN,2026-04-02T00:00:43.082Z,2,"[{'isbn10': '', 'isbn13': '9798350430479'}]","[{'name': 'Amazon', 'url': 'https://www.amazon..."


In [80]:
df_prepared.shape

(3866, 28)

Попробуем найти рейтинги для книг, взятых с NY Times. Это поможет нам сравнить их с рейтингами с литреса. Мы воспользовались ресурсом Open Library, где оценки ставятся похожим образом

In [81]:
import requests
import logging
import time

logging.config.fileConfig('logging.ini', defaults={'logfilename': 'openlibrary.log'})

In [28]:
logger = logging.getLogger("sLogger")

Пример получения книги "О дивный новый мир" (вставил ее, а не 1984, так как 1984 - попса). В headers рекомендуется указывать контактные данные: http://openlibrary.org/developers/api#:~:text=limiting%20or%20blocking.-,Rate%20Limits,enjoy%20a%203x%20request%20limit.&text=Example%20Requests:,Python%20example%20request

Будем искать по названию и автору и выбирать лишь один подходящий ответ

В запросе ниже мы смогли получить id произведения на open library, с него мы сможем найти рейтинги

In [ ]:
user = input()
email = input()

In [ ]:
url = 'https://openlibrary.org/search.json'
headers = {'User-Agent': f'Korney Sanockin (korney@yandex.ru)'}
title = 'Brave New World'
author = 'Aldous Huxley'
params = {'title': title, 'author': author, 'fields': 'title,key', 'limit': 1}

page = requests.get(url, params=params, headers=headers).json()
key = page['docs'][0]['key']
key

'/works/OL64365W'

In [30]:
ratings = requests.get(f'https://openlibrary.org{key}/ratings.json').json()
ratings

{'summary': {'average': 3.9715536105032823,
  'count': 457,
  'sortable': 3.8867770442659264},
 'counts': {'1': 9, '2': 22, '3': 96, '4': 176, '5': 154}}

Напишем функцию где на вход программа будет получать автора и название книги, а на выход - средний рейтинг и количество оценок

In [ ]:
def get_ratings(title, author, user='Korney Sanochkin', email='sanochkin@yandex.ru'):
    url = 'https://openlibrary.org/search.json'
    headers = {'User-Agent': f'{user} ({email})', 'accept': 'application/json'}
    params = {'title': title, 'author': author, 'fields': 'author_name,title,key', 'limit': 1}

    try:
        page = requests.get(url, params=params, headers=headers).json()
        time.sleep(1)
        key = page['docs'][0]['key']
        real_title = page['docs'][0]['title']
        real_author = page['docs'][0]['author_name']
        logger.info(f'Книга {title} найдена на open-library')
        ratings = requests.get(f"https://openlibrary.org{key}/ratings.json").json()
        avg_rating = ratings['summary']['average']
        rating_count = ratings['summary']['count']
        logger.info(f'Рейтинги к книге {title} найдены')
        res = {'avg_rating': avg_rating, 'rating_count': rating_count, 'real_title': real_title, 'real_author': real_author}
        return res
    except Exception as e:
        logger.exception(e)
        return None

А теперь попробуем найти все книги на Open Library. При указании контактов мы можем делать 3 запроса в секунду, поэтому на всякий случай будем тормозить программу на 1 секунду каждые три запроса

In [87]:
start = int(input())
end = int(input())

In [88]:
user = input()
email = input()

In [ ]:

ratings = {'author': [], 'title': [], 'avg_rating': [], 'rating_count': [], 'title_open_library': [], 'author_open_library': []}
for i, row in df_prepared.iloc[start:end].iterrows():
    time.sleep(2)
    title, author = row['title'], row['author']
    ratings['title'].append(title)
    ratings['author'].append(author)
    open_library_raw = get_ratings(title, author, user=user, email=email)
    if open_library_raw:
        avg_rating = open_library_raw['avg_rating']
        rating_count = open_library_raw['rating_count']
        real_title = open_library_raw['real_title']
        real_author = open_library_raw['real_author']
    else:
        avg_rating = None
        rating_count = None
        real_title = None
        real_author = None

    ratings['avg_rating'].append(avg_rating)
    ratings['rating_count'].append(rating_count)
    ratings['title_open_library'].append(real_title)
    ratings['author_open_library'].append(real_author)

In [57]:
open_library_df = pd.DataFrame(ratings)

In [58]:
open_library_df

,author,title,avg_rating,rating_count,title_open_library,author_open_library
0,Navessa Allen,LIGHTS OUT,4.409091,22,Lights Out,[Navessa Allen]


In [59]:
merged_df = pd.merge(df_prepared, open_library_df, on=['author', 'title'], how='left')

In [ ]:
merged_df.to_csv('open_library_x_nyt.csv')